In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('/content/sample_data/train.csv')

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11017 entries, 0 to 11016
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ApplicationDate             10487 non-null  object 
 1   Age                         10487 non-null  float64
 2   AnnualIncome                10487 non-null  float64
 3   CreditScore                 9986 non-null   float64
 4   LoanAmount                  9986 non-null   float64
 5   LoanDuration                10487 non-null  float64
 6   MaritalStatus               10487 non-null  object 
 7   NumberOfDependents          10487 non-null  float64
 8   HomeOwnershipStatus         10487 non-null  object 
 9   MonthlyDebtPayments         9986 non-null   float64
 10  CreditCardUtilizationRate   10487 non-null  float64
 11  NumberOfOpenCreditLines     10487 non-null  float64
 12  NumberOfCreditInquiries     10487 non-null  float64
 13  DebtToIncomeRatio           104

In [4]:
data.head()

,ApplicationDate,Age,AnnualIncome,CreditScore,LoanAmount,LoanDuration,MaritalStatus,NumberOfDependents,HomeOwnershipStatus,MonthlyDebtPayments,...,EmploymentStatus,EducationLevel,Experience,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,LoanApproved,RiskScore
0,2010-06-26,27.0,66829.0,549.0,17290.0,60.0,Divorced,1.0,Rent,1095.0,...,Employed,Associate,4.0,35067.0,0.257790,0.251465,508.970230,0.288013,0.0,66.176500
1,1996-09-23,55.0,172147.0,850.0,16110.0,36.0,Widowed,1.0,Mortgage,211.0,...,Employed,High School,33.0,27001.0,0.086110,0.093173,514.675859,0.050585,1.0,28.495737
2,2015-01-19,51.0,300000.0,850.0,38436.0,36.0,Married,0.0,Mortgage,546.0,...,Employed,Bachelor,28.0,278382.0,0.108436,0.115443,1268.276385,0.072571,1.0,34.488104
3,1981-05-12,25.0,34683.0,847.0,19186.0,48.0,Married,0.0,Other,153.0,...,Employed,High School,0.0,9224.0,0.100686,0.112822,498.505187,0.225415,1.0,36.910753
4,1995-05-07,55.0,300000.0,850.0,30437.0,48.0,Single,2.0,Rent,562.0,...,Employed,Bachelor,31.0,4502.0,0.110437,0.089037,756.035156,0.052721,1.0,31.347091


In [5]:
data = data.dropna()

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for column in data.drop(['ApplicationDate'], axis=1).columns:
    if data[column].dtype == 'object':
        data[column] = le.fit_transform(data[column])

In [7]:
Q1 = data['RiskScore'].quantile(0.25)
Q3 = data['RiskScore'].quantile(0.75)
IQR = Q3 - Q1
data = data[~((data['RiskScore'] < (Q1 - 1.5 * IQR)) | (data['RiskScore'] > (Q3 + 1.5 * IQR)))]

corr_matrix = data.drop(['ApplicationDate'], axis=1).corr()
high_corr = corr_matrix[(corr_matrix >= 0.7) & (corr_matrix != 1.)].stack()
high_corr

Age                 Experience            0.982958
AnnualIncome        MonthlyIncome         0.984978
                    LoanApproved          0.742338
LoanAmount          MonthlyLoanPayment    0.872616
TotalAssets         NetWorth              0.993996
MonthlyIncome       AnnualIncome          0.984978
                    LoanApproved          0.754733
Experience          Age                   0.982958
NetWorth            TotalAssets           0.993996
BaseInterestRate    InterestRate          0.975344
                    RiskScore             0.758712
InterestRate        BaseInterestRate      0.975344
                    RiskScore             0.742940
MonthlyLoanPayment  LoanAmount            0.872616
LoanApproved        AnnualIncome          0.742338
                    MonthlyIncome         0.754733
RiskScore           BaseInterestRate      0.758712
                    InterestRate          0.742940
dtype: float64

In [8]:
to_drop = ['ApplicationDate', 'RiskScore', 'AnnualIncome', 'Experience', 'InterestRate', 'CreditScore', 'TotalAssets', 'LoanAmount']
X = data.drop(to_drop, axis=1)
y = data['RiskScore']

In [9]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

LinearRegression()

In [10]:
test_data = pd.read_csv('/content/sample_data/test.csv')

In [11]:
for column in test_data.drop(['ApplicationDate'], axis=1).columns:
    if test_data[column].dtype == 'object':
        test_data[column] = le.fit_transform(test_data[column])

In [12]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          5000 non-null   int64  
 1   ApplicationDate             5000 non-null   object 
 2   Age                         5000 non-null   int64  
 3   AnnualIncome                5000 non-null   int64  
 4   CreditScore                 5000 non-null   int64  
 5   LoanAmount                  5000 non-null   int64  
 6   LoanDuration                5000 non-null   int64  
 7   MaritalStatus               5000 non-null   int64  
 8   NumberOfDependents          5000 non-null   int64  
 9   HomeOwnershipStatus         5000 non-null   int64  
 10  MonthlyDebtPayments         5000 non-null   int64  
 11  CreditCardUtilizationRate   5000 non-null   float64
 12  NumberOfOpenCreditLines     5000 non-null   int64  
 13  NumberOfCreditInquiries     5000 

In [13]:
from sklearn.impute import SimpleImputer
to_drop = ['ApplicationDate', 'ID', 'AnnualIncome', 'Experience', 'InterestRate', 'CreditScore', 'TotalAssets', 'LoanAmount']


X_test = test_data.drop(to_drop, axis=1)
y_test = model.predict(X_test)

In [14]:
y_test

array([32.49334262, 58.44374396, 29.7657769 , ..., 63.73631175,
       51.76683247, 66.10607291])

In [15]:
ids = range(5000)
df = pd.DataFrame({'ID': ids, 'RiskScore': y_test})
df.to_csv('submission.csv', index=False)